# 03 — Reescrita: de paciente→médico para médico→médico

**Entrada:** `../data/corpus_curado.parquet` (saída do `02`).
**Saída:** `../data/corpus_reescrito.parquet`.

O MedPT vem do Doctoralia: **paciente perguntando a médico**, em linguagem leiga e relato
pessoal. O assistente que estamos construindo atende o inverso — **médico consultando um
assistente clínico**. Sem esta etapa, treinaríamos o modelo na distribuição errada.

Notebook caro: faz chamada de API por linha. Tem cache em disco e um piloto obrigatório
antes da execução completa.

---

## Por que a reescrita vem antes do split

Na versão anterior o split acontecia primeiro e a reescrita rodava só sobre o treino.
Isso tinha dois problemas:

- A reescrita pode devolver `descartar`. Rodando depois do split, os descartes furam a
  estratificação que acabou de ser montada.
- Se só o treino muda de registro, treina-se numa distribuição e avalia-se em outra.

Aqui a reescrita passa no corpus inteiro; o split vem depois, no `04`.

In [3]:
from pathlib import Path
import json

import pandas as pd

DATA = Path.cwd().parent / "data"
CORPUS_CURADO = DATA / "corpus_curado.parquet"
CORPUS_REESCRITO = DATA / "corpus_reescrito.parquet"
CACHE = DATA / "reescrita_cache.jsonl"
SEED = 42

corpus = pd.read_parquet(CORPUS_CURADO)
print(f"{len(corpus):,} linhas a reescrever | {corpus['condition'].nunique()} condições")
corpus.head(3)

17,239 linhas a reescrever | 651 condições


,id,question,answer,condition,medical_specialty,question_type,cluster_id,cluster_size,tam_q,tam_a,origem,fonte
0,1612,Meu filho apresentou bruxismo noturno e após u...,"Não, não é normal, o leve para avaliação com p...",Bruxismo,Pediatra,Diagnóstico,1471,1,160,61,medpt,AKCIT/MedPT
1,1613,Meu filho apresentou bruxismo noturno e após u...,É possível que o bruxismo diurno esteja relaci...,Bruxismo,Pediatra,Diagnóstico,1472,1,160,708,medpt,AKCIT/MedPT
2,1701,Meu bebê tem 6 meses tem 4 dentes ele fica rig...,"Sim, a primeira consulta com dentista deve ser...",Bruxismo,Pediatra,Escolha de profissionais de saúde,1473,1,104,94,medpt,AKCIT/MedPT


## Decisão — quando reescrever a resposta, e o que isso custa

Reescrever a **pergunta** é barato e seguro: a pergunta é chave de recuperação, o conteúdo
clínico dela não precisa estar correto para o dataset prestar.

Reescrever a **resposta** é outra coisa. Os tokens-alvo deixam de ser resposta de médico
real e passam a ser saída do `glm-5.3-flash`. Na prática o dataset vira, em parte,
destilação de um modelo pequeno — e qualquer alucinação introduzida aqui vira ground truth
no fine-tuning. **Isso precisa estar declarado no relatório.**

Por isso reescrever a resposta não é automático: é uma ação separada, e a **última**
escolha. As quatro ações, em ordem de preferência:

| ação | pergunta | resposta |
|---|---|---|
| `manter` | original | original |
| `reescrever_pergunta` | reescrita | **original** |
| `reescrever_ambos` | reescrita | reescrita |
| `descartar` | — linha sai — | |

`reescrever_pergunta` existe porque o caso mais comum do MedPT é pergunta leiga com
resposta que já é impessoal e clínica. Aí não há motivo para tocar no alvo: preserva-se
texto de médico como ground truth de graça, sem custar chamada a mais.

O limite dela é real e o prompt precisa cobri-lo. As respostas do MedPT não estão em
registro de paciente, mas muitas estão em registro **médico→paciente** — "leve seu filho
para avaliação", "agende sua consulta". Casar isso com uma pergunta em registro técnico
produz um par incoerente, que é exatamente o que esta etapa deveria estar consertando.
Então `reescrever_pergunta` só vale quando a resposta original já é impessoal; fora disso,
`reescrever_ambos`.

Os guarda-corpos:

1. O prompt restringe a reescrita da resposta a **registro e coerência**, com proibição
   explícita de adicionar ou remover conteúdo clínico.
2. A ação `descartar` pode disparar por problema na **resposta** — antes o modelo só via a
   pergunta e não tinha como julgar isso.
3. Quando a ação preserva a resposta, o texto original é **reinjetado em código**; o eco do
   modelo não é usado. Paráfrase silenciosa não passa (ver `reescreve`).
4. Verificação automática de números e unidades sobre as respostas reescritas.
5. Juiz de fidelidade sobre uma amostra, para estimar a taxa de desvio com um número.

**Botões desta etapa:** o texto do `SYSTEM_REESCRITA`, `TRUNCA_RESPOSTA_ENTRADA` e o
tamanho do piloto.

In [4]:
import os
from dotenv import load_dotenv
from openai import AsyncOpenAI

load_dotenv()
if not os.getenv("Z_API_KEY"):
    raise RuntimeError("Z_API_KEY ausente. Defina no .env (veja .env.example).")

client = AsyncOpenAI(
    api_key=os.getenv("Z_API_KEY"),
    base_url="https://api.z.ai/api/paas/v4/",
)

MODELO_REESCRITA = "glm-5.3-flash"
CONCORRENCIA = 50

In [5]:
from enum import Enum
from pydantic import BaseModel, Field

class Acao(str, Enum):
    REESCREVER_AMBOS = "reescrever_ambos"
    REESCREVER_PERGUNTA = "reescrever_pergunta"
    MANTER = "manter"
    DESCARTAR = "descartar"

# Quais acoes preservam o texto original de cada lado. Usados para reinjetar o
# original em vez de confiar no eco do modelo, e para marcar as linhas depois.
ACOES_PERGUNTA_ORIGINAL = {Acao.MANTER.value, Acao.DESCARTAR.value}
ACOES_RESPOSTA_ORIGINAL = {Acao.REESCREVER_PERGUNTA.value, Acao.MANTER.value, Acao.DESCARTAR.value}
ACOES_APROVEITADAS = {Acao.REESCREVER_AMBOS.value, Acao.REESCREVER_PERGUNTA.value, Acao.MANTER.value}

class Reescrita(BaseModel):
    acao: Acao = Field(description='reescrever_ambos, reescrever_pergunta, manter ou descartar')
    pergunta: str = Field(description='pergunta no registro medico-medico; a original quando manter ou descartar')
    resposta: str = Field(description='resposta no registro tecnico quando reescrever_ambos; a original nas demais acoes')
    motivo: str = Field(description='justificativa curta, em uma frase')


def formato_esperado(modelo: type[BaseModel]) -> str:
    """Renderiza o modelo como TEMPLATE de saida, nao como JSON Schema.

    Colar o `model_json_schema()` cru no prompt faz o modelo copiar o schema
    inteiro na resposta e anexar os campos reais no fim. O template abaixo
    mostra so as chaves, os valores de enum e a descricao de cada campo.
    """
    schema = modelo.model_json_schema()
    defs = schema.get("$defs", {})
    campos = {}
    for nome, prop in schema["properties"].items():
        if "$ref" in prop:
            prop = defs[prop["$ref"].split("/")[-1]] | prop
        if "enum" in prop:
            campos[nome] = " | ".join(str(v) for v in prop["enum"])
        else:
            campos[nome] = f"<{prop.get('description', prop['type'])}>"
    return json.dumps(campos, ensure_ascii=False, indent=2)


SYSTEM_REESCRITA = (
    "Voce adapta pares de pergunta e resposta medicas para o registro de comunicacao "
    "entre profissionais de saude, em um hospital maternidade.\n\n"
    "O material original vem de uma plataforma onde PACIENTES perguntam e MEDICOS "
    "respondem. Voce transforma isso em um par como seria entre um medico e um "
    "assistente clinico.\n\n"
    "REGRA CENTRAL, acima de todas as outras: nao adicione nem remova conteudo clinico. "
    "Voce muda registro, estrutura e pessoa do discurso. Voce NAO acrescenta diagnostico, "
    "dose, criterio, exame ou conduta que nao esteja no texto original. Se o original "
    "e vago, a reescrita continua vaga.\n\n"
    "SEGUNDA REGRA: a resposta original foi escrita por um medico real e vale mais que "
    "qualquer reescrita sua. Preserve-a sempre que ela puder ser preservada. Reescreva a "
    "resposta apenas quando mante-la deixaria o par incoerente.\n\n"
    "Escolha uma acao, nesta ordem de preferencia:\n\n"
    "manter: o par ja esta em registro tecnico adequado dos dois lados. Devolva pergunta "
    "e resposta sem alteracao.\n\n"
    "reescrever_pergunta: a pergunta esta em registro leigo, MAS a resposta ja e "
    "impessoal e clinica — nao se dirige ao paciente em segunda pessoa, nao manda agendar "
    "consulta nem procurar especialista, nao tem saudacao nem despedida, nao fala do caso "
    "pessoal de quem perguntou. Reescreva so a pergunta (entre 8 e 40 palavras, terminando "
    "em interrogacao, sobre conduta, criterio ou manejo) de modo que a resposta original "
    "continue respondendo a ela exatamente. Devolva a resposta original sem tocar nela. "
    "Esta e a acao preferida sempre que a resposta permitir: verifique se ela permite "
    "ANTES de considerar reescrever_ambos.\n\n"
    "reescrever_ambos: a pergunta esta em registro leigo E a resposta nao pode ser "
    "preservada, porque se dirige ao paciente ('voce deve', 'leve seu filho', 'agende sua "
    "consulta'), tem saudacao ou despedida, ou so faz sentido como fala para leigo. "
    "Reescreva a pergunta como acima e a resposta no mesmo registro tecnico, respondendo "
    "exatamente a pergunta reescrita, preservando todo o conteudo clinico do original e "
    "apenas ele.\n\n"
    "descartar: o par nao serve. Use quando:\n"
    "- a pergunta pedir interpretacao de um valor de exame especifico do paciente;\n"
    "- a pergunta ou a resposta estiver truncada ou incompreensivel;\n"
    "- a resposta nao responder a pergunta;\n"
    "- a resposta for apenas um encaminhamento generico ('procure um especialista') "
    "sem conteudo clinico;\n"
    "- a resposta for propaganda de consultorio ou contato comercial.\n"
    "Ao descartar, devolva os textos originais e explique no motivo.\n\n"
    "Responda apenas o JSON abaixo preenchido, exatamente com estas quatro chaves, "
    "sem texto antes ou depois:\n"
    f"{formato_esperado(Reescrita)}"
)

In [6]:
import asyncio
import random
import re

def parse_json(texto):
    limpo = re.sub(r"^```(?:json)?\s*|\s*```$", "", texto.strip()).strip()
    return json.loads(limpo)

# O 02 ja cortou respostas acima do p99 (~1600 chars), entao na pratica nada e truncado
# aqui. O limite fica como rede: se um dia entrar texto maior, o modelo julga o registro
# por um prefixo, mas o original preservado continua sendo o texto inteiro.
TRUNCA_RESPOSTA_ENTRADA = 3000   # chars de answer enviados ao modelo
sem = asyncio.Semaphore(CONCORRENCIA)
uso_tokens = []                  # acumula usage para estimar custo

async def reescreve(row, tentativas=4):
    user = (
        f"Condicao: {row['condition']}\n"
        f"Pergunta original: {row['question']}\n"
        f"Resposta original: {str(row['answer'])[:TRUNCA_RESPOSTA_ENTRADA]}"
    )
    base = {"id": int(row["id"]), "question_original": row["question"],
            "answer_original": row["answer"]}
    async with sem:
        for tentativa in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model=MODELO_REESCRITA,
                    messages=[
                        {"role": "system", "content": SYSTEM_REESCRITA},
                        {"role": "user", "content": user},
                    ],
                    response_format={
                        "type": "json_schema",
                        "json_schema": {"name": "Reescrita", "schema": Reescrita.model_json_schema(), "strict": True},
                    },
                    temperature=0.3,
                )
                if resp.usage:
                    uso_tokens.append((resp.usage.prompt_tokens, resp.usage.completion_tokens))
                r = parse_json(resp.choices[0].message.content)
                acao = r["acao"]
                # Onde a acao preserva o texto, reinjetamos o original em vez de aceitar o
                # eco do modelo. O eco costuma vir com parafrase leve, espaco a mais ou
                # acento trocado -- o suficiente para a linha deixar de ser texto de medico
                # real sem que nada no pipeline acuse.
                pergunta = row["question"] if acao in ACOES_PERGUNTA_ORIGINAL else r["pergunta"]
                resposta = row["answer"] if acao in ACOES_RESPOSTA_ORIGINAL else r["resposta"]
                return base | {
                    "acao": acao,
                    "question": pergunta,
                    "answer": resposta,
                    "motivo": r.get("motivo", ""),
                }
            except Exception as erro:
                if tentativa == tentativas - 1:
                    return base | {"acao": "erro", "question": row["question"],
                                   "answer": row["answer"],
                                   "motivo": f"{type(erro).__name__}: {erro}"}
                await asyncio.sleep(2 ** tentativa + random.random())

## Piloto obrigatório

Respostas chegam a 3000 chars contra ~100 das perguntas, então o custo desta etapa é
uma ordem de grandeza acima da reescrita só-de-pergunta. Rodar o piloto, olhar o
resultado e a estimativa **antes** de comprometer a execução completa.

In [7]:
N_PILOTO = 30
TRUNCA_PRINT = 500   # chars de resposta mostrados; suba se quiser ler inteiro

piloto = corpus.sample(N_PILOTO, random_state=SEED)
print(f"{N_PILOTO} chamadas | modelo {MODELO_REESCRITA} | concorrência {CONCORRENCIA}")

# as_completed em vez de gather: cada par aparece assim que volta, na ordem em que
# volta. Com gather a tela fica parada ate a ultima chamada terminar.
# Prefixos: "-" antes, "+" depois, "=" texto que a acao preservou (byte a byte,
# reinjetado pelo `reescreve`) -- o "=" e evidencia, nao rotulo declarado pelo modelo.
resultados_piloto = []
for futuro in asyncio.as_completed([reescreve(r) for _, r in piloto.iterrows()]):
    r = await futuro
    resultados_piloto.append(r)

    print("\n" + "=" * 92)
    print(f"[{len(resultados_piloto)}/{N_PILOTO}] id={r['id']}  {r['acao'].upper()}")
    if r["motivo"]:
        print(f"  motivo: {r['motivo']}")

    if r["question"] == r["question_original"]:
        print(f"\n  P =  {r['question_original']}")
    else:
        print(f"\n  P -  {r['question_original']}")
        print(f"  P +  {r['question']}")

    if r["answer"] == r["answer_original"]:
        print(f"\n  R =  {str(r['answer'])[:TRUNCA_PRINT]}", flush=True)
    else:
        print(f"\n  R -  {str(r['answer_original'])[:TRUNCA_PRINT]}")
        print(f"  R +  {str(r['answer'])[:TRUNCA_PRINT]}", flush=True)

piloto_df = pd.DataFrame(resultados_piloto)
print("\n" + "=" * 92)
print(piloto_df["acao"].value_counts().to_string())

if uso_tokens:
    entrada = sum(t[0] for t in uso_tokens) / len(uso_tokens)
    saida = sum(t[1] for t in uso_tokens) / len(uso_tokens)
    print(f"\nmédia por chamada: {entrada:.0f} tokens entrada, {saida:.0f} saída")
    print(f"projeção para {len(corpus):,} linhas: "
          f"{entrada*len(corpus)/1e6:.1f}M entrada, {saida*len(corpus)/1e6:.1f}M saída")

30 chamadas | modelo glm-5.3-flash | concorrência 50

[1/30] id=831572  REESCREVER_PERGUNTA
  motivo: A resposta já é impessoal e clínica, sem saudação ou encaminhamento, respondendo diretamente à questão; apenas a pergunta leiga foi convertida ao registro técnico.

  P -  a pomada para candidíase pode interromper a menstruação?
  P +  Cremes vaginais antifúngicos utilizados no tratamento de candidíase podem interferir no eixo hormonal a ponto de interromper o fluxo menstrual?

  R =  os cremes vaginais agem somente no local e não teriam ação sobre o eixo hormonal, a ponto de interromper o fluxo menstrual.

[2/30] id=862333  DESCARTAR
  motivo: A resposta está truncada e incoerente ('disso acontecer' não se refere a nada da pergunta), não responde ao questionamento sobre indicação cirúrgica e se reduz a um encaminhamento genérico sem conteúdo clínico.

  P =  Meu mioma está 94x83Medida uterina 138 x 111 x 96Sinto muitas dores muito fluxo já tenho 4 filhosSeria indicado cirurgia

  R = 

In [8]:
# Inspecao qualitativa. Duas perguntas diferentes por acao:
#  - reescrever_ambos: o registro mudou sem que o conteudo clinico mudasse?
#  - reescrever_pergunta: a pergunta nova casa com a resposta que foi preservada?
print("=" * 80)
print("REESCREVER_AMBOS — conferir que nada clinico entrou ou saiu")
for r in piloto_df[piloto_df["acao"] == "reescrever_ambos"].head(3).itertuples():
    print("-" * 80)
    print(f"P antes : {r.question_original[:200]}")
    print(f"P depois: {r.question[:200]}")
    print(f"R antes : {str(r.answer_original)[:280]}")
    print(f"R depois: {str(r.answer)[:280]}")

print("\n" + "=" * 80)
print("REESCREVER_PERGUNTA — conferir que a resposta preservada responde a pergunta nova")
for r in piloto_df[piloto_df["acao"] == "reescrever_pergunta"].head(4).itertuples():
    print("-" * 80)
    print(f"P antes : {r.question_original[:200]}")
    print(f"P depois: {r.question[:200]}")
    print(f"R (preservada): {str(r.answer)[:280]}")

print("\n" + "=" * 80 + "\nDESCARTADOS")
for r in piloto_df[piloto_df["acao"] == "descartar"].head(5).itertuples():
    print(f"- {r.motivo}  ::  {r.question_original[:80]}")

REESCREVER_AMBOS — conferir que nada clinico entrou ou saiu
--------------------------------------------------------------------------------
P antes : Tenho mioma e faço uso da injeção sinto alguns sintomas como insônia isso é normal?
P depois: Paciente com mioma em uso de medicação injetável (provavelmente hormonal) relata insônia: esse sintoma pode ser atribuído à medicação?
R antes : Não dá para afirmar que a insônia seja causada por uma injeção que provavelmente deve ser hormonal sem saber o nome da injeção .Por favor, verifique qual e a injeção para te dar uma resposta certa.
R depois: Não é possível afirmar que a insônia seja causada pela injeção, que provavelmente deve ser hormonal, sem saber o nome da medicação. É necessário identificar qual injeção está em uso para dar uma resposta correta.
--------------------------------------------------------------------------------
P antes : Gostaria de saber quais são as alternativas para preservação da fertilidade durante o tratamento d

## Verificação de fidelidade

Duas checagens complementares, cobrindo as duas populações:

**As respostas preservadas** (`manter`, `reescrever_pergunta`) têm uma garantia forte e
barata: precisam ser byte-idênticas ao original. Como o `reescreve` reinjeta o texto
original em vez de aceitar o eco do modelo, isso é uma invariante — o `assert` abaixo
existe para detectar se alguém quebrar essa reinjeção depois.

**As respostas reescritas** (`reescrever_ambos`) não têm garantia nenhuma, só evidência.
Números e unidades são a parte concretamente verificável de uma resposta clínica: dose,
idade gestacional, limiar de exame. Se a reescrita **introduz** um número que não estava no
original, ela inventou conteúdo.

Não é prova de fidelidade — paráfrase pode distorcer sem mexer em número algum. É o piso
barato, complementado pelo juiz da célula seguinte. E note que a nova ação
`reescrever_pergunta` encolhe a população que depende desse piso: quanto mais linhas caem
nela, menos superfície de destilação o dataset tem.

In [9]:
NUM = re.compile(r"\d+(?:[.,]\d+)?")

def compara_fidelidade(original, reescrito):
    orig = set(NUM.findall(str(original)))
    novo = set(NUM.findall(str(reescrito)))
    return {
        "num_adicionados": sorted(novo - orig),
        "num_removidos": sorted(orig - novo),
        "razao_tamanho": len(str(reescrito)) / max(len(str(original)), 1),
    }

def preservadas_intactas(df):
    """Invariante: onde a acao preserva a resposta, ela e byte-identica ao original."""
    sub = df[df["acao"].isin(ACOES_RESPOSTA_ORIGINAL)]
    divergentes = sub[sub["answer"] != sub["answer_original"]]
    assert divergentes.empty, (
        f"{len(divergentes)} respostas marcadas como preservadas divergem do original "
        f"(ids: {divergentes['id'].head(5).tolist()})"
    )
    return len(sub)

def audita(df):
    """Diff de numeros -- so faz sentido onde a resposta de fato mudou."""
    linhas = []
    for r in df.itertuples():
        if r.acao != Acao.REESCREVER_AMBOS.value:
            continue
        checagem = compara_fidelidade(r.answer_original, r.answer)
        linhas.append({"id": r.id, **checagem})
    return pd.DataFrame(linhas)

n_preservadas = preservadas_intactas(piloto_df)
print(f"OK — {n_preservadas} respostas preservadas, byte-idênticas ao original")

auditoria = audita(piloto_df)
if len(auditoria):
    com_adicao = auditoria[auditoria["num_adicionados"].str.len() > 0]
    print(f"\n{len(com_adicao)}/{len(auditoria)} respostas reescritas introduziram número "
          f"novo ({len(com_adicao)/len(auditoria):.0%})")
    print(f"razão de tamanho: mediana {auditoria['razao_tamanho'].median():.2f}, "
          f"máx {auditoria['razao_tamanho'].max():.2f}")
    for r in com_adicao.head(5).itertuples():
        print(f"  id={r.id} adicionou {r.num_adicionados}")
else:
    print("\nnenhuma resposta reescrita no piloto para auditar")

OK — 8 respostas preservadas, byte-idênticas ao original

0/22 respostas reescritas introduziram número novo (0%)
razão de tamanho: mediana 0.92, máx 1.38


### Juízes (amostra)

Cada ação tem um modo de falha próprio, e um juiz para ele. Rodam sobre amostra: o objetivo
é produzir uma taxa para o relatório, não filtrar linha a linha.

**`reescrever_ambos` → juiz de fidelidade.** O diff de números não pega distorção
semântica. Este juiz compara original e reescrita procurando afirmação clínica que tenha
sido adicionada, removida ou invertida.

**`reescrever_pergunta` → juiz de coerência.** Modo de falha novo, criado por esta ação: a
resposta é texto de médico intacto, mas a pergunta reescrita pode ter deslizado de assunto,
e aí o par ensina o modelo a responder outra coisa. Preservar a resposta elimina o risco de
alucinação e cria este — que é mais barato de detectar, porque não exige julgar conteúdo
clínico, só se a resposta responde a pergunta.

In [10]:
class Fiel(int, Enum):
    SIM = 1
    NAO = 0

class Fidelidade(BaseModel):
    fiel: Fiel = Field(description='1 se a reescrita preserva as afirmacoes clinicas do original, 0 caso contrario')
    problema: str = Field(description='o desvio em uma frase, ou vazio se fiel=1')

class Coerencia(BaseModel):
    coerente: Fiel = Field(description='1 se a resposta responde a pergunta reescrita, 0 caso contrario')
    problema: str = Field(description='o descasamento em uma frase, ou vazio se coerente=1')


SYSTEM_FIDELIDADE = (
    "Voce compara uma resposta medica original com sua versao reescrita.\n\n"
    "Responda fiel=1 se a reescrita preserva exatamente as afirmacoes clinicas do "
    "original: mesmos diagnosticos, mesmas condutas, mesmas ressalvas, mesma forca de "
    "recomendacao. Mudanca de registro, ordem ou concisao NAO torna infiel.\n\n"
    "Responda fiel=0 se a reescrita adiciona afirmacao clinica ausente do original, "
    "remove ressalva relevante, ou inverte o sentido de uma recomendacao.\n\n"
    "Responda apenas o JSON abaixo preenchido, exatamente com estas duas chaves, "
    "sem texto antes ou depois:\n"
    f"{formato_esperado(Fidelidade)}"
)

SYSTEM_COERENCIA = (
    "Voce recebe uma PERGUNTA reescrita e a RESPOSTA original que ela deveria endereçar. "
    "A resposta nao foi alterada; so a pergunta foi.\n\n"
    "Responda coerente=1 se a resposta responde a pergunta: o assunto e o mesmo e a "
    "resposta cobre o que foi perguntado. Resposta parcial ou mais ampla que a pergunta "
    "ainda e coerente.\n\n"
    "Responda coerente=0 se a pergunta reescrita mudou de assunto, se pergunta algo que a "
    "resposta nao aborda, ou se pressupoe informacao que a resposta contradiz.\n\n"
    "Responda apenas o JSON abaixo preenchido, exatamente com estas duas chaves, "
    "sem texto antes ou depois:\n"
    f"{formato_esperado(Coerencia)}"
)


async def _julga(row, system, modelo, chave, tentativas=3):
    if chave == "fiel":
        user = (f"ORIGINAL:\n{str(row['answer_original'])[:2000]}\n\n"
                f"REESCRITA:\n{str(row['answer'])[:2000]}")
    else:
        user = (f"PERGUNTA REESCRITA:\n{row['question']}\n\n"
                f"RESPOSTA ORIGINAL:\n{str(row['answer'])[:2000]}")
    async with sem:
        for tentativa in range(tentativas):
            try:
                resp = await client.chat.completions.create(
                    model=MODELO_REESCRITA,
                    messages=[{"role": "system", "content": system},
                              {"role": "user", "content": user}],
                    response_format={"type": "json_schema",
                                     "json_schema": {"name": modelo.__name__, "schema": modelo.model_json_schema(), "strict": True}},
                    temperature=0,
                )
                r = parse_json(resp.choices[0].message.content)
                return {"id": int(row["id"]), chave: int(r[chave]), "problema": r.get("problema", "")}
            except Exception as erro:
                if tentativa == tentativas - 1:
                    return {"id": int(row["id"]), chave: None, "problema": str(erro)}
                await asyncio.sleep(2 ** tentativa + random.random())


async def roda_juiz(df, acao, system, modelo, chave, rotulo):
    sub = df[df["acao"] == acao]
    if not len(sub):
        print(f"{rotulo}: nenhuma linha para julgar")
        return None
    vereditos = pd.DataFrame(await asyncio.gather(
        *(_julga(r, system, modelo, chave) for _, r in sub.iterrows())
    ))
    taxa = vereditos[chave].mean()
    print(f"{rotulo}: {taxa:.0%} ({int(vereditos[chave].sum())}/{len(vereditos)})")
    for r in vereditos[vereditos[chave] == 0].head(5).itertuples():
        print(f"  id={r.id}: {r.problema}")
    return vereditos

veredito_fid = await roda_juiz(piloto_df, Acao.REESCREVER_AMBOS.value,
                               SYSTEM_FIDELIDADE, Fidelidade, "fiel",
                               "fidelidade (reescrever_ambos)")
veredito_coe = await roda_juiz(piloto_df, Acao.REESCREVER_PERGUNTA.value,
                               SYSTEM_COERENCIA, Coerencia, "coerente",
                               "coerência (reescrever_pergunta)")

fidelidade (reescrever_ambos): 73% (16/22)
  id=715949: A reescrita removeu a recomendação clínica de conversar com o ginecologista, que era parte da conduta original.
  id=336387: A reescrita remove a ressalva 'sempre consulte com o seu obstetra', eliminando a recomendação de acompanhamento profissional presente no original, embora mantenha os dados diagnósticos e o limiar de 92 mg/dL.
  id=395409: A reescrita enfraquece a força da recomendação: o original sugere 'iniciar anticoagulação com enoxaparina', enquanto a reescrita diz apenas 'considerar o início' da anticoagulação.
  id=715969: A reescrita remove as recomendações de sempre seguir as orientações do médico, agendar a consulta de reavaliação e esclarecer dúvidas, condutas presentes no original.
  id=386861: A reescrita omite as orientações finais do original: agendar consulta de reavaliação, esclarecer dúvidas, revisar o caso e conversar com o médico, removendo condutas relevantes.
coerência (reescrever_pergunta): 100% (6/6)


## Execução completa

Só depois de olhar o piloto. O cache é por `id` e a célula é retomável: se a execução
cair no meio, rodar de novo continua de onde parou em vez de recomeçar do zero.

In [11]:
from collections import Counter

RODAR_COMPLETO = True   # vire para True depois de aprovar o piloto
AVISA_A_CADA = 25        # de quantas em quantas linhas imprime o andamento

def carrega_cache():
    if not CACHE.exists():
        return {}
    return {r["id"]: r for r in
            (json.loads(l) for l in CACHE.read_text(encoding="utf-8").splitlines() if l.strip())}

feitos = carrega_cache()
print(f"cache: {len(feitos):,} de {len(corpus):,} linhas")

if RODAR_COMPLETO:
    pendentes = corpus[~corpus["id"].isin(feitos)]
    LOTE = 500   # so limita quantas chamadas ficam vivas ao mesmo tempo
    print(f"processando {len(pendentes):,} pendentes "
          f"(concorrência {CONCORRENCIA}, lotes de {LOTE})\n")

    placar, n = Counter(), 0
    with CACHE.open("a", encoding="utf-8") as f:
        for inicio in range(0, len(pendentes), LOTE):
            bloco = pendentes.iloc[inicio:inicio + LOTE]
            # as_completed entrega uma a uma, na ordem em que voltam. Cada linha e
            # gravada e sincronizada na hora: interromper a execucao perde apenas o
            # que estiver em voo, nao o lote inteiro. Com gather, o primeiro write
            # so aconteceria depois das 500 chamadas do lote.
            for futuro in asyncio.as_completed([reescreve(r) for _, r in bloco.iterrows()]):
                reg = await futuro
                f.write(json.dumps(reg, ensure_ascii=False) + "\n")
                f.flush()
                placar[reg["acao"]] += 1
                n += 1
                if n % AVISA_A_CADA == 0 or n == len(pendentes):
                    print(f"  {n:>6,}/{len(pendentes):,} ({n/len(pendentes):>4.0%})  "
                          + "  ".join(f"{a}={c}" for a, c in placar.most_common()),
                          flush=True)
    feitos = carrega_cache()
    print(f"\ncache final: {len(feitos):,}")
else:
    print("RODAR_COMPLETO=False — nada executado.")

cache: 14,655 de 17,239 linhas
processando 2,584 pendentes (concorrência 50, lotes de 500)



      25/2,584 (  1%)  reescrever_ambos=20  reescrever_pergunta=5
      50/2,584 (  2%)  reescrever_ambos=39  reescrever_pergunta=11
      75/2,584 (  3%)  reescrever_ambos=56  reescrever_pergunta=16  descartar=3
     100/2,584 (  4%)  reescrever_ambos=80  reescrever_pergunta=17  descartar=3
     125/2,584 (  5%)  reescrever_ambos=98  reescrever_pergunta=20  descartar=7
     150/2,584 (  6%)  reescrever_ambos=114  reescrever_pergunta=23  descartar=13
     175/2,584 (  7%)  reescrever_ambos=133  reescrever_pergunta=25  descartar=17
     200/2,584 (  8%)  reescrever_ambos=153  reescrever_pergunta=26  descartar=21
     225/2,584 (  9%)  reescrever_ambos=174  reescrever_pergunta=28  descartar=23
     250/2,584 ( 10%)  reescrever_ambos=194  reescrever_pergunta=32  descartar=24
     275/2,584 ( 11%)  reescrever_ambos=214  reescrever_pergunta=36  descartar=25
     300/2,584 ( 12%)  reescrever_ambos=234  reescrever_pergunta=41  descartar=25
     325/2,584 ( 13%)  reescrever_ambos=250  reescrev

In [12]:
if not feitos:
    raise RuntimeError("Cache vazio. Rode a célula anterior com RODAR_COMPLETO=True.")

reescrito_df = pd.DataFrame(feitos.values())
print(reescrito_df["acao"].value_counts().to_string())
print(f"\ndescartados: {(reescrito_df['acao'] == 'descartar').sum():,} "
      f"({(reescrito_df['acao'] == 'descartar').mean():.1%})")
print(f"erros: {(reescrito_df['acao'] == 'erro').sum():,}")

# O numero que vai para o relatorio: quanto do alvo de treino continua sendo
# texto escrito por medico, e nao saida do glm-5.3-flash.
aprov = reescrito_df[reescrito_df["acao"].isin(ACOES_APROVEITADAS)]
humanas = aprov["acao"].isin(ACOES_RESPOSTA_ORIGINAL)
print(f"\nrespostas humanas originais entre as aproveitadas: {humanas.mean():.1%} "
      f"({int(humanas.sum()):,}/{len(aprov):,})")

acao
reescrever_ambos       12951
reescrever_pergunta     2308
descartar               1939
manter                    41

descartados: 1,939 (11.2%)
erros: 0

respostas humanas originais entre as aproveitadas: 15.4% (2,349/15,300)


In [13]:
# Auditoria sobre TODAS as linhas do cache (o diff barato, nao o juiz).
n_preservadas = preservadas_intactas(reescrito_df)
print(f"OK — {n_preservadas:,} respostas preservadas, byte-idênticas ao original")

auditoria_total = audita(reescrito_df)
if len(auditoria_total):
    com_adicao = auditoria_total[auditoria_total["num_adicionados"].str.len() > 0]
    print(f"\nrespostas reescritas com número novo: {len(com_adicao):,}/"
          f"{len(auditoria_total):,} ({len(com_adicao)/len(auditoria_total):.1%})")
    print(f"razão de tamanho mediana: {auditoria_total['razao_tamanho'].median():.2f}")
else:
    print("\nnenhuma resposta reescrita no cache")

OK — 4,288 respostas preservadas, byte-idênticas ao original

respostas reescritas com número novo: 416/12,951 (3.2%)
razão de tamanho mediana: 0.96


Merge de volta no corpus curado, preservando as colunas de curadoria (`cluster_id`,
`cluster_size`, `origem`, `fonte`) e guardando os textos originais para auditoria.
As linhas `descartar` e `erro` saem aqui.

O flag booleano `reescrito` dá lugar a dois: `pergunta_reescrita` e `resposta_reescrita`.
Só o segundo marca destilação — é ele que responde "quanto do alvo de treino não é texto
de médico", que é a pergunta que o relatório precisa responder.

In [14]:
aproveitados = reescrito_df[reescrito_df["acao"].isin(ACOES_APROVEITADAS)]

final = corpus.drop(columns=["question", "answer"]).merge(
    aproveitados[["id", "question", "answer", "question_original",
                  "answer_original", "acao"]],
    on="id", how="inner",
)
final["pergunta_reescrita"] = final["acao"].isin(
    [Acao.REESCREVER_AMBOS.value, Acao.REESCREVER_PERGUNTA.value])
final["resposta_reescrita"] = final["acao"] == Acao.REESCREVER_AMBOS.value

# Invariante que segue para o 04: onde resposta_reescrita e False, o alvo de treino
# e literalmente o texto do medico. Se isso quebrar, o numero do relatorio mente.
intactas = final.loc[~final["resposta_reescrita"]]
assert (intactas["answer"] == intactas["answer_original"]).all(), \
    "resposta marcada como preservada difere do original"

final.to_parquet(CORPUS_REESCRITO, index=False)
print(f"{len(corpus):,} → {len(final):,} linhas "
      f"({len(final)/len(corpus):.0%} aproveitado)")
print(f"pergunta reescrita: {final['pergunta_reescrita'].mean():.0%} | "
      f"resposta reescrita: {final['resposta_reescrita'].mean():.0%} "
      f"({1 - final['resposta_reescrita'].mean():.0%} dos alvos são texto humano)")
print(f"{final['cluster_id'].nunique():,} clusters | "
      f"{final['condition'].nunique()} condições")
print(f"→ {CORPUS_REESCRITO.name}")

17,239 → 15,300 linhas (89% aproveitado)
pergunta reescrita: 100% | resposta reescrita: 85% (15% dos alvos são texto humano)
10,195 clusters | 616 condições
→ corpus_reescrito.parquet
